# Exp 40: Stacking without ConvNeXt-Tiny

Card: [exp40_stacking_no_convnext.md](../../experiments/exp40_stacking_no_convnext.md)


In [ ]:
import os

import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import precision_recall_curve

from core.config import CLASSES, ROOT
from core.cv import PROB_COLS, VIP, Config
from core.scores import Scores
from core.submission import Ensemble, Stacking

SMOKE = os.environ.get("SMOKE") == "1"
CONFIG_DIR = ROOT / "sections" / "model" / "configs"
MODELS = {
    "exp26 ResNet-18": Config.from_toml(
        CONFIG_DIR / "exp26_coarse_dropout_5fold" / "resnet18.toml"
    ),
    "EfficientNet-B0": Config.from_toml(
        CONFIG_DIR / "exp32_efficientnet_b0_5fold" / "efficientnet_b0.toml"
    ),
    "RegNetY-016": Config.from_toml(
        CONFIG_DIR / "exp33_regnety_016" / "regnety_016.toml"
    ),
    "ConvNeXt-Tiny": Config.from_toml(
        CONFIG_DIR / "exp34_convnext_tiny" / "convnext_tiny.toml"
    ),
}
WITHOUT = ["exp26 ResNet-18", "EfficientNet-B0", "RegNetY-016"]  # ConvNeXt-Tiny dropped
if SMOKE:
    MODELS = {
        name: MODELS[name]
        for name in ["exp26 ResNet-18", "EfficientNet-B0", "ConvNeXt-Tiny"]
    }
    WITHOUT = ["exp26 ResNet-18", "EfficientNet-B0"]
C = 0.1  # exp39 conclusion
MARGIN = 0.05  # plateau: F1 >= max - MARGIN (exp37, exp38)
FIGURES = ROOT / "sections" / "model" / "figures"
PREFIX = "exp40_stacking_no_convnext"
SAVE_FIGURES = not SMOKE

# dataviz reference palette, light mode
INK, PEAK, MIDDLE = "#52514e", "#eb6834", "#2a78d6"
SERIES = {"with ConvNeXt-Tiny": "#eb6834", "without ConvNeXt-Tiny": "#1baf7a"}
plt.rcParams.update(
    {
        "figure.facecolor": "#fcfcfb",
        "axes.facecolor": "#fcfcfb",
        "savefig.facecolor": "#fcfcfb",
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.edgecolor": INK,
        "xtick.color": INK,
        "ytick.color": INK,
        "axes.titlesize": "medium",
        "axes.titlecolor": INK,
        "axes.grid": True,
        "grid.color": "#e1e0d9",
        "legend.frameon": False,
        "legend.fontsize": "small",
    }
)

## Stacking with and without ConvNeXt-Tiny


In [ ]:
oofs = dict(zip(MODELS, Ensemble.member_oofs(MODELS.values())))
labels, folds = (
    oofs[WITHOUT[0]][CLASSES].to_numpy(),
    oofs[WITHOUT[0]]["fold"].to_numpy(),
)
stacked, weights = {}, {}
for name, members in {
    "with ConvNeXt-Tiny": list(MODELS),
    "without ConvNeXt-Tiny": WITHOUT,
}.items():
    features = np.stack(
        [Stacking.logit(oofs[m][PROB_COLS].to_numpy()) for m in members], axis=1
    )  # rows, members, classes
    stacked[name], weights[name] = Stacking.cross_fold(features, labels, folds, C, members)

## F1 against threshold


In [ ]:
fig, axes = plt.subplots(
    1, len(CLASSES), figsize=(3.6 * len(CLASSES), 3.6), sharey=True
)
for c, (cls, ax) in enumerate(zip(CLASSES, axes)):
    for name, probs in stacked.items():
        precision, recall, thr = precision_recall_curve(labels[:, c], probs[:, c])
        f1 = (2 * precision * recall / (precision + recall + 1e-12))[:-1]
        ax.plot(thr, f1, color=SERIES[name], lw=2, label=name)
    ticks = [0, 0.2, 0.4, 0.6, 0.8, 1]
    ax.set(
        title=cls,
        xlabel="threshold",
        xlim=(0, 1),
        ylim=(0, 1.05),
        xticks=ticks,
        yticks=ticks,
    )
    ax.tick_params(labelleft=True, labelsize=8)
axes[0].set_ylabel("F1")
fig.suptitle(
    f"F1 against threshold, stacking C = {C}, out-of-fold rows of folds 0–4", color=INK
)
fig.tight_layout(rect=(0, 0, 1, 0.88))
fig.legend(
    *axes[0].get_legend_handles_labels(),
    loc="upper center",
    ncols=2,
    bbox_to_anchor=(0.5, 0.93),
)
figures = {f"{PREFIX}_f1_curves.png": fig}
plt.show()

## Weights without ConvNeXt-Tiny


In [ ]:
# coefficient per member logit and intercept, mean and std over 5 fits; fold = held-out fold of that fit
w = weights["without ConvNeXt-Tiny"]
display(w.groupby("class", sort=False).agg(["mean", "std"]).round(2))

## Peak vs plateau without ConvNeXt-Tiny


In [ ]:
# stacking without ConvNeXt-Tiny, VIP: peak vs plateau-middle threshold (exp39 layout)
probs = stacked["without ConvNeXt-Tiny"]
peak, plateau = Scores(labels, probs).best_f1(), Scores(labels, probs).plateau_f1(MARGIN)
fig, axes = plt.subplots(1, len(VIP), figsize=(4.2 * len(VIP), 3.6), sharey=True)
for cls, ax in zip(VIP, axes):
    c = CLASSES.index(cls)
    precision, recall, thr = precision_recall_curve(labels[:, c], probs[:, c])
    f1 = (2 * precision * recall / (precision + recall + 1e-12))[:-1]
    ax.axvspan(
        plateau.loc[cls, "low"],
        plateau.loc[cls, "high"],
        color=MIDDLE,
        alpha=0.12,
        lw=0,
        label=f"F1 within {MARGIN} of max",
    )
    ax.plot(thr, f1, color=INK, lw=1.5)
    best_t, mean_t = peak.loc[cls, "threshold"], plateau.loc[cls, "threshold"]
    ax.axvline(best_t, color=PEAK, lw=2, ls="--", label="best_t (peak)")
    ax.axvline(mean_t, color=MIDDLE, lw=2, label="mean_t (plateau middle)")
    # value labels at two heights, outward from the pair so they never cross a line; both left near right edge
    for t, label, y in [(best_t, "best_t", 0.22), (mean_t, "mean_t", 0.08)]:
        right = t == max(best_t, mean_t) and t < 0.8
        ax.text(
            t + 0.015 if right else min(best_t, mean_t) - 0.015,
            y,
            f"{label} {t:.3f}",
            ha="left" if right else "right",
            fontsize=8,
            color=INK,
        )
    ticks = [0, 0.2, 0.4, 0.6, 0.8, 1]
    ax.set(
        title=cls,
        xlabel="threshold",
        xlim=(0, 1),
        ylim=(0, 1.05),
        xticks=ticks,
        yticks=ticks,
    )
    ax.tick_params(labelleft=True, labelsize=8)
axes[0].set_ylabel("without ConvNeXt-Tiny\nF1")
fig.suptitle(
    f"F1 against threshold, stacking without ConvNeXt-Tiny, C = {C}, out-of-fold rows of folds 0–4",
    color=INK,
)
fig.tight_layout(rect=(0, 0, 1, 0.88))
fig.legend(
    *axes[0].get_legend_handles_labels(),
    loc="upper center",
    ncols=3,
    bbox_to_anchor=(0.5, 0.93),
)
figures[f"{PREFIX}_plateau.png"] = fig
plt.show()

In [ ]:
if SAVE_FIGURES:
    for fname, fig in figures.items():
        fig.savefig(FIGURES / fname, dpi=120)